In [10]:
import os
import glob
import random
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ---------------- Configuration ---------------- #

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

MAX_LENGTH = 256
NUM_SAMPLES = 1000
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device :", device)
print("GPU :", torch.cuda.get_device_name(0))

# ---------------- Locate Dataset ---------------- #

json_files = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)

if len(json_files) == 0:
    raise Exception("No JSONL dataset found.")

DATA_PATH = json_files[0]

print("\nDataset :", DATA_PATH)

df = pd.read_json(DATA_PATH, lines=True)

print("\nOriginal Shape :", df.shape)

df = df[["instruction","context","response"]].fillna("")

df = df.sample(NUM_SAMPLES, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED
)

print(f"Train : {len(train_df)}")
print(f"Validation : {len(valid_df)}")
print(f"Test : {len(test_df)}")

Device : cuda
GPU : Tesla T4

Dataset : /kaggle/input/datasets/vyshnavsb/dataset/databricks-dolly-15k.jsonl

Original Shape : (15015, 4)
Train : 800
Validation : 100
Test : 100


In [11]:
def create_prompt(row):

    if row["context"].strip():

        return f"""### Instruction:
{row['instruction']}

### Context:
{row['context']}

### Response:
{row['response']}"""

    else:

        return f"""### Instruction:
{row['instruction']}

### Response:
{row['response']}"""


train_df["text"] = train_df.apply(create_prompt, axis=1)
valid_df["text"] = valid_df.apply(create_prompt, axis=1)
test_df["text"] = test_df.apply(create_prompt, axis=1)

train_dataset = Dataset.from_pandas(train_df[["text"]])
valid_dataset = Dataset.from_pandas(valid_df[["text"]])
test_dataset = Dataset.from_pandas(test_df[["text"]])

print(train_dataset)

print("\nExample:\n")

print(train_dataset[0]["text"])

Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 800
})

Example:

### Instruction:
Which is a species of fish? Bream or Cream

### Response:
Bream


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading Tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

print("Loading Model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model.config.pad_token_id = tokenizer.pad_token_id

print(f"\nParameters : {model.num_parameters():,}")

Loading Tokenizer...
Loading Model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Parameters : 361,821,120


In [13]:
from transformers import DefaultDataCollator
from transformers import DataCollatorForLanguageModeling

def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


train_dataset = train_dataset.map(tokenize_function)
valid_dataset = valid_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

train_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(train_dataset)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['text', '__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 800
})


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./SmolLM2_Dolly",

    do_train=True,
    do_eval=True,

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=2e-5,

    weight_decay=0.01,

    fp16=True,
    bf16=False,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=25,

    save_strategy="steps",
    save_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    report_to="none",

    seed=42,

    optim="adamw_torch"
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=True,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=25,
eval_strategy=IntervalStrategy.STEPS,
eval_us

In [15]:
from transformers import Trainer
from transformers import EarlyStoppingCallback

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("Trainer Ready")

Trainer Ready


In [16]:
print(model.dtype)

torch.float32


In [17]:
print("Starting Training...\n")

train_result = trainer.train()

print("\nTraining Completed!")

trainer.save_state()

Starting Training...



Step,Training Loss,Validation Loss
25,2.192135,2.237049
50,2.023039,2.217882
75,2.114391,2.209118
100,2.159633,2.204855
125,2.026739,2.200315
150,2.010691,2.196855
175,1.978084,2.194880
200,2.007794,2.194385


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



Training Completed!


In [22]:
import os

SAVE_PATH = "./SmolLM2_Dolly_Final"

os.makedirs(SAVE_PATH, exist_ok=True)

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("✅ Model saved successfully!")
print("Location:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved successfully!
Location: ./SmolLM2_Dolly_Final


In [23]:
import os

print(os.path.exists("/kaggle/working/SmolLM2_Dolly_Finetuned"))

print(os.listdir("/kaggle/working"))

False
['bloomz_model', 'SmolLM2_Dolly_Final.zip', 'SmolLM2_Dolly_Final', '.virtual_documents', 'SmolLM2_Dolly']


In [25]:
import os

print(os.listdir("/kaggle/working/SmolLM2_Dolly_Final"))

['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'generation_config.json', 'chat_template.jinja']


In [19]:
import shutil

ZIP_NAME = "SmolLM2_Dolly_Final"

shutil.make_archive(
    ZIP_NAME,
    "zip",
    SAVE_PATH
)

print("✅ Zip file created!")
print(f"{ZIP_NAME}.zip")

✅ Zip file created!
SmolLM2_Dolly_Final.zip


In [26]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_PATH = "/kaggle/working/SmolLM2_Dolly_Final"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

print("Loaded Successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded Successfully!


In [27]:
prompt = """### Instruction:
What is Machine Learning?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

### Instruction:
What is Machine Learning?

### Response:
Machine Learning is a branch of computer science that involves developing software systems that can automatically learn from data. It is a subset of artificial intelligence. 

Machine learning systems are able to learn from data without being explicitly programmed. The learning can be done through supervised, unsupervised, or reinforcement learning. 

The goal of machine learning is to build systems that can make predictions and decisions based on data. This can be done through supervised, unsupervised, or reinforcement learning. 

Supervised learning is when the system is given labeled data and its output is used to train the system. Unsupervised learning is when
